In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.svm import SVC

from classical import CKAResCNet
from quantum import get_quantum_circuit, inference_quantum_hardware
from models.quantum_models import QKAResQNet

In [2]:
def load_and_preprocess(file_path="datasets/fingertips/fingertips-dataset.csv"):
    df = pd.read_csv(file_path, sep=";")

    # 1. Isolate continuous gesture blocks
    df["block"] = (df["state"] != df["state"].shift(1)).cumsum()
    blocks = (
        df[df["state"] != "unlabeled"]
        .groupby(["block", "state"])
        .size()
        .reset_index(name="count")
    )

    # 2. Drop the dead sensor (#1) and system columns
    active_cols = [
        c for c in df.columns if c not in ["time", "state", "block"] and "1" not in c
    ]

    X_list, y_list = [], []
    for _, row in blocks.iterrows():
        block_data = df[df["block"] == row["block"]][active_cols].values

        # 3. Extract temporal geometric features (flatten the trajectory)
        features = np.concatenate(
            [
                np.mean(block_data, axis=0),
                np.std(block_data, axis=0),
                np.min(block_data, axis=0),
                np.max(block_data, axis=0),
            ]
        )
        X_list.append(features)
        y_list.append(row["state"])

    X = np.array(X_list)
    y = LabelEncoder().fit_transform(np.array(y_list))

    return X, y, len(np.unique(y))

In [3]:
def train_pytorch(model, X_train, y_train, epochs=100, lr=0.005, verbose=True):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    X_t = torch.FloatTensor(X_train)
    y_t = torch.LongTensor(y_train)

    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        out = model(X_t)
        loss = criterion(out, y_t)
        loss.backward()
        optimizer.step()

        if verbose and (epoch + 1) % 25 == 0:
            print(f"    Epoch {epoch + 1:3d}/{epochs} | Loss: {loss.item():.4f}")
    return model

In [4]:
# --- Data Prep ---
X, y, num_classes = load_and_preprocess("datasets/fingertips/fingertips-dataset.csv")
print(
    f"Data Ready: {X.shape[0]} samples, {X.shape[1]} features, {num_classes} classes."
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=43, stratify=y
)

scaler = MinMaxScaler(feature_range=(0, np.pi))
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_tensor = torch.FloatTensor(X_train)
X_test_tensor = torch.FloatTensor(X_test)
y_train_tensor = torch.LongTensor(y_train)
y_test_tensor = torch.LongTensor(y_test)

# Hyperparameters
epochs = 100
lr = 0.005
num_qubits = 4
hidden_dim = num_qubits

print(f"Using num_qubits={num_qubits}, epochs={epochs}, lr={lr}")

Data Ready: 56 samples, 96 features, 10 classes.
Using num_qubits=4, epochs=100, lr=0.005


In [7]:
# 1. Classical RBF-SVM
print("\nTraining Classical SVM...")
svm = SVC(kernel="rbf", C=10, gamma="scale")
svm.fit(X_train, y_train)
svm_acc = accuracy_score(y_test, svm.predict(X_test))
print(f"[Result] Classical SVM: {svm_acc * 100:.1f}%")


Training Classical SVM...
[Result] Classical SVM: 92.9%


In [ ]:
# 2. CKAResCNet
print("\nTraining CKAResCNet...")
cka = CKAResCNet(input_dim=X.shape[1], output_dim=num_classes, hidden_dim=hidden_dim)
cka = train_pytorch(cka, X_train, y_train, epochs=epochs, lr=lr, verbose=True)

with torch.no_grad():
    cka_preds = torch.argmax(cka(X_test_tensor), dim=1).numpy()
cka_acc = accuracy_score(y_test, cka_preds)
print(f"[Result] CKAResCNet: {cka_acc * 100:.1f}%")


Training CKAResCNet...
    Epoch  25/100 | Loss: 0.2399
    Epoch  50/100 | Loss: 0.0597
    Epoch  75/100 | Loss: 0.0262
    Epoch 100/100 | Loss: 0.0147
[Result] CKAResCNet: 85.7%


In [ ]:
# 3. QKAResQNet on the simulator
print("\nTraining QKAResQNet...")
qnn, feature_map, ansatz, observables = get_quantum_circuit(num_qubits)
qka = QKAResQNet(
    qnn=qnn,
    input_dim=X.shape[1],
    output_dim=num_classes,
    num_qubits=num_qubits,
)
qka = train_pytorch(qka, X_train, y_train, epochs=epochs, lr=lr, verbose=True)

with torch.no_grad():
    qka_preds = torch.argmax(qka(X_test_tensor), dim=1).numpy()
qka_acc = accuracy_score(y_test, qka_preds)
print(f"[Result] QKAResQNet: {qka_acc * 100:.1f}%")


Training QKAResQNet...
Setting up BackendEstimatorV2...
* HARDWARE REALITY: IBM Kingston (Heron r2) Simulation Active
   • Gate Noise: 1q=0.02%, 2q=0.20%
   • Readout Noise: 1.15% (Dominant Factor)
Transpiling circuit to backend basis gates...
Setting up Simulator...
    Epoch  25/100 | Loss: 0.2380
    Epoch  50/100 | Loss: 0.0730
    Epoch  75/100 | Loss: 0.0417
    Epoch 100/100 | Loss: 0.0287
[Result] QKAResQNet: 85.7%



In [10]:
print("\n==================================")
print("=== FINAL RESULTS (single split) ===")
print("==================================")
print(f"SVM        : {svm_acc * 100:5.2f}%")
print(f"CKAResCNet : {cka_acc * 100:5.2f}%")
print(f"QKAResQNet : {qka_acc * 100:5.2f}%")


=== FINAL RESULTS (single split) ===
SVM        : 92.86%
CKAResCNet : 85.71%
QKAResQNet : 85.71%


In [ ]:
accuracy_qh, inference_time_qh, predictions_qh = inference_quantum_hardware(
    qka,
    feature_map,
    ansatz,
    observables,
    X_test_tensor,
    y_test_tensor,
)

print("-" * 30)
print("       Quantum Hardware MODEL RESULTS       ")
print("-" * 30)
print(f"Accuracy:           {accuracy_qh * 100:.2f}%")
print(f"Inference Time:     {inference_time_qh:.5f} sec")
print("-" * 30)

qiskit_runtime_service._discover_account:WARNING:2026-05-25 20:10:20,200: Loading account with the given token. A saved account will not be used.



--- Connecting to IBM Quantum Hardware(ibm_quantum_platform) ---


qiskit_runtime_service.__init__:WARNING:2026-05-25 20:10:24,332: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-05-25 20:10:24,333: Using instance: open-instance, plan: open


Selected Backend: ibm_kingston
Transpiling circuit...


No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


Injecting hardware circuit into PyTorch model...
Submitting 14 samples to ibm_kingston... (This may take a while)
Hardware Inference Complete. Accuracy: 92.86%
------------------------------
       Quantum Hardware MODEL RESULTS       
------------------------------
Accuracy:           92.86%
Inference Time:     84.07402 sec
------------------------------
